# 직접 수집 5 · 보고서 — AI 요약

수집·분석 결과(통계)를 **LLM이 해석**해 보고서를 자동 작성합니다.
- 로컬 vLLM(OpenAI 호환) **Qwen3-8B** 사용, 엔드포인트 `http://localhost:8005/v1`
- ⚠️ vLLM 서버가 켜져 있어야 합니다. (WSL: `bash /home/sumin/boot_qwen8b_vllm.sh`)

In [ ]:
# --- 공통 준비: 프로젝트 루트로 이동 + 임포트 + 경로 정의 ---
import os, sys, warnings
warnings.filterwarnings("ignore")
for _ in range(5):                       # wiki_crawling.py 있는 프로젝트 루트 탐색
    if os.path.isfile("wiki_crawling.py"):
        break
    os.chdir("..")
ROOT = os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
import pandas as pd
import pipeline as P
import wiki_crawling as wc

SEED = "Quantum computing"               # ← 분석할 위키 문서 (자유 변경)
N_DEPTH = 1                              # 1차수 고정
slug = P.slugify_seed(SEED)
BASE = os.path.join("runs", slug)
os.makedirs(os.path.join(BASE, "seed_item"), exist_ok=True)
os.makedirs(os.path.join(BASE, "xtools_item", slug), exist_ok=True)
EXPAND   = os.path.join(BASE, "seed_item", f"{N_DEPTH}차시 확장 최종_결과.xlsx")
FILTER   = os.path.join(BASE, "seed_item", f"{slug}_filtering_network.xlsx")
XTOOLS   = os.path.join(BASE, "xtools_item", slug)
FLAG     = os.path.join(XTOOLS, ".collect_done")
PAGERANK = os.path.join(BASE, f"{slug}_pagerank.xlsx")
STATS    = os.path.join(BASE, f"{slug}_statistics.xlsx")
print("작업 루트:", os.getcwd())
print("대상 문서:", SEED, "| 작업 폴더:", BASE)

## 통계 결과 로드 (04 노트북 산출물)

In [ ]:
if not os.path.isfile(STATS):
    raise FileNotFoundError("통계 파일이 없습니다. 04 노트북을 먼저 실행하세요.")
stats = pd.read_excel(STATS)
# 기술집약도 기준 상위 아이템을 보고서 근거로
cols = [c for c in ["title", "기술집약도", "수요부상성", "공급부상성"] if c in stats.columns]
top = stats[cols].sort_values("기술집약도", ascending=False).head(12)
rows = top.to_dict("records")
print("보고서 근거 상위", len(rows), "건")
top

## LLM 호출 → 보고서

In [ ]:
import json
from openai import OpenAI
client = OpenAI(base_url="http://localhost:8005/v1", api_key="EMPTY", timeout=90)

system = ("당신은 KISTI의 기술기획 분석가입니다. 위키 네트워크에서 수집한 지표 데이터를 근거로 "
          "교육생이 이해하기 쉬운 한국어 유망아이템 보고서를 작성합니다. 데이터에 없는 수치는 지어내지 말고, "
          "기술집약도·수요/공급 부상성의 의미를 풀어 해석하세요. 마크다운으로 간결하게.")
user = (f"시드 문서: {SEED}\n\n[연관 문서 지표 상위 (JSON)]\n"
        f"{json.dumps(rows, ensure_ascii=False, indent=2)}\n\n"
        "다음 보고서를 작성하세요:\n## 1. 분석 개요\n## 2. 주목 아이템 TOP 3 해석\n## 3. 종합 시사점")

resp = client.chat.completions.create(
    model="Qwen/Qwen3-8B",
    messages=[{"role": "system", "content": system}, {"role": "user", "content": user}],
    temperature=0.4, max_tokens=1800,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)
print(resp.choices[0].message.content)

**정리** — 위키 원본 크롤링부터 수집→정제→결과→보고서까지 전 과정을 코드로 시연했습니다. 🎉